In [1]:
#Import libs
import pandas as pd
import numpy as np
from openai import OpenAI
import os
from anthropic import Anthropic
import json
from google import genai

In [5]:
from dotenv import load_dotenv

load_dotenv() 


True

Loading csv file with pilot candidates for translating and importing in SrpWN

In [20]:
#Load srpwn excel file
pilot_file = "./data/nedostajuci_sinsetovi.csv"

pilot_data = pd.read_csv(pilot_file)
pilot_data.shape

(200, 12)

In [21]:
pilot_data.head()

,domain,BCS,ID,literals,def,usage,hypernymID,hypernym_literals,hypernym_def,hypernym_usage,Prevod literala,Prevod def
0,medicine,NaN,ENG30-14327543-n,histamine headache; cluster headache,a painful recurring headache associated with t...,-,ENG30-14326607-n,headache; head ache; cephalalgia,pain in the head caused by dilation of cerebra...,-,NaN,NaN
1,medicine,NaN,ENG30-14289942-n,scorch; singe,a surface burn,-,ENG30-14289590-n,burn,an injury cause by exposure to heat or chemica...,-,NaN,NaN
2,medicine,NaN,ENG30-09782855-n,alexic,a person with alexia,-,ENG30-10405694-n,patient,a person who requires medical care,1 : the number of emergency patients has grown...,NaN,NaN
3,medicine,NaN,ENG30-02380980-v,chicken out; back off; pull out; back down; bo...,remove oneself from an obligation,1 : He bowed out when he heard how much work w...,ENG30-01766952-v,retire; withdraw,lose interest,1 : he retired from life when his wife died,NaN,NaN
4,medicine,NaN,ENG30-14159318-n,clinodactyly,a congenital defect in which one or more toes ...,-,ENG30-14465048-n,birth defect; congenital anomaly; congenital d...,a defect that is present at birth,-,NaN,NaN


In [22]:
#Load srpwn excel file
# check_file = "./data/SrpWN_translated_synsets.csv"

# check_file = pd.read_csv(check_file)
# check_file.shape

We are removing last 2 columns because they don't exist in SrpWN

In [23]:
candidates = pilot_data[pilot_data.columns[:-2]]

#### Prompt strategies: 
- Zero-shot
- Few-shot
- Chain-of-thought
- Prompt sa kontekstom
- Više kandidata

#### Models:
- GPT
- Claude
- Gemini


Define differnet model versions

In [24]:
ai_models = {
    'gpt': {'gpt-4o', 'gpt-5.5'},
    'claude': {'claude-opus-4-8', 'claude-sonnet-5'}
}

models = ai_models.keys()
versions = ai_models.values()

### API wrapper functions - GPT, Claude & Gemini
 In this section we define 3 helper fuctions to interact with different LLM providers:
 - `call_gpt(model, prompt)` - Sends request(prompt) to the OpenAI GPT API
 - `call_claude(model, prompt)` - Sends request(prompt) to the Anthropic Claude API
 - `call_gemini(prompt)` - Send request(prompt) to the Google Gemini API
 - model parameter defines different API version

In [25]:
# GPT
def call_gpt(model, prompt):

    client = OpenAI(
        api_key = os.environ.get("OPENAI_API_KEY"))
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    
    result = response.choices[0].message.content
    
    return result

In [26]:
#CLAUDE
def call_claude(model, prompt):

    client = Anthropic(
        # This is the default and can be omitted
        api_key=os.environ.get("ANTHROPIC_API_KEY"))
    
    message = client.messages.create(
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model=model,
    )
    
    for block in message.content:
        if block.type == "text":
            return block.text



In [33]:
#GEMINI - it

client = genai.Client(
    api_key = os.environ.get("GEMINI_API_KEY")
)

# 3. Generate content
response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents='Tell me a one-liner joke about coding.',
)

print(response.output_text)


 



ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

`format_prompt(row)` function receives 2 parameters - prompt strategy template and one record from WordNet. It formats the final prompt by substituting template pleaceholders with values extracted from the WordNet row.

In [28]:
def get_prompt(template, row):
    prompt = template.format(
        domain = row["domain"],
        literals = row["literals"],
        literal_id = row["ID"],
        definition = row['def'],
        usage = row["usage"],
        hypernymID = row["hypernymID"],
        hypernym_literals = row["hypernym_literals"],
        hypernym_def = row["hypernym_def"],
        hypernym_usage = row["hypernym_usage"] 
    )	

    return prompt

### Zero-shot strategy

#### Load Zero-shot prompt strategy template 

In [29]:
with open("./prompts/zero_shot.txt", "r", encoding="utf-8") as f:
    zero_shot_template = f.read()

zero_shot_template

'You are an expert translator in the medical field, specialized in Serbian language and medical terminology.\nYour task is to translate a Princeton WordNet synsets from English to Serbian. All synsets are related to medical terms.\nPurpose of this tranlsation is expanding Serbian WordNet (SrpWN) in the medical domain.\n\nRequirements:\n- Use Serbian medical terminology used in academic papers.\n- Translate the literal(s), definition and usage example.\n- If a standard Serbian medical term exists, prefer it over a descriptive translation.\n- Do not translate following columns - BCS, ID and hypernymID.\n- Do not add explanations or comments.\n\nInput synset: \nDomain: {domain}\nLiterals: {literals}\nID: {literal_id}\nDefinition: {definition}\nUsage: {usage}\nHypernymID: {hypernymID} \nHypernym_literals: {hypernym_literals}                     \nHypernym_def: {hypernym_def}         \nHypernym_usage : {hypernym_usage}   \n\nReturn ONLY valid JSON in the following format:\n{{\n  "domain" : 

#### 1.1. Testing Zero-shot strategy on just one synset

In [30]:
row = pilot_data.iloc[0]
row

domain                                                        medicine
BCS                                                                NaN
ID                                                    ENG30-14327543-n
literals                          histamine headache; cluster headache
def                  a painful recurring headache associated with t...
usage                                                                -
hypernymID                                            ENG30-14326607-n
hypernym_literals                     headache; head ache; cephalalgia
hypernym_def         pain in the head caused by dilation of cerebra...
hypernym_usage                                                       -
Prevod literala                                                    NaN
Prevod def                                                         NaN
Name: 0, dtype: object

#### 1.2. Calling model API-s and testing zero-shot strategy

In [31]:
prompt = get_prompt(zero_shot_template, row)

for model, versions in ai_models.items():
    for version in versions:        
        match model:
            case 'gpt':
                print("GPT model version ", version, call_gpt(version, prompt))
            case 'claude':
                print("Claude model version ", version, call_claude(version, prompt))
            # case 'gemini':
                # TODO

    

GPT model version  gpt-4o ```json
{
  "domain": "medicina",
  "BCS": "",
  "ID": "ENG30-14327543-n",
  "literals": "histaminska glavobolja; klaster glavobolja",
  "def": "bolna, ponavljajuća glavobolja povezana sa oslobađanjem histamina iz ćelija",
  "usage": "-",
  "hypernymID": "ENG30-14326607-n",
  "hypernym_literals": "glavobolja; bol u glavi; cefalalgija",
  "hypernym_def": "bol u glavi izazvan širenjem moždanih arterija, kontrakcijama mišića ili reakcijom na lekove",
  "hypernym_usage": "-"
}
```
GPT model version  gpt-5.5 {
  "domain": "medicina",
  "BCS": "",
  "ID": "ENG30-14327543-n",
  "literals": "histaminska glavobolja; klaster glavobolja",
  "def": "bolna recidivirajuća glavobolja povezana sa oslobađanjem histamina iz ćelija",
  "usage": "-",
  "hypernymID": "ENG30-14326607-n",
  "hypernym_literals": "glavobolja; cefalalgija",
  "hypernym_def": "bol u glavi izazvan dilatacijom cerebralnih arterija, mišićnim kontrakcijama ili reakcijom na lekove",
  "hypernym_usage": "-"
}

#### 1.3. First results and thoughts

After evaluating different AI models using zero-shot prompt strategy based on the results from **JUST ONE** synset - ENG30-14327543-n - histamine headache; cluster headache - first conclusions are:

 - OpenAI models versions gpt-5 and gpt-5.5 gave similar and mostly valid translations. There are some punctuational differences and different choice of synonyms (rekurentna, ponvaljajuca).
 - OpenAI models gave wrong translation of a tehnical term (strucni naziv?) used in Serbian scientific literature (cefalalgija instead of cefalgija) ❌ 
  Anthropic Claude translated the term correctly ✅ (note: later in the testing is noticed that this model also sometimes return bad term) 
 - I'll give Claude model slight addvantage over Gpt for now, altough Gpt answer used complexed terrminology and sound more academic.


Next steps include testing models on bigger set and different prompt strategies.


#### 2.1. Testing Zero-shot strategy on 20 synsets, all models

In [ ]:
zero_shot_result_file_path = "./results/zero_shot_20.xlsx" 

folder_path = os.path.dirname(zero_shot_result_file_path)
if folder_path and not os.path.exists(folder_path):
    os.makedirs(folder_path)

# 2. Create an empty Excel file if it does not exist
if not os.path.exists(zero_shot_result_file_path):
    # Create an empty DataFrame and save it as an Excel file
    df_empty = pd.DataFrame()
    df_empty.to_excel(zero_shot_result_file_path, index=False)
    print(f"Successfully created empty file at: {zero_shot_result_file_path}")
else:
    print(f"File already exists at: {zero_shot_result_file_path}")

In [ ]:
## MULTIPLE SYNSETS


rows = pilot_data.iloc[0:5]
header = True

with pd.ExcelWriter(zero_shot_result_file_path, mode="a", if_sheet_exists="overlay", engine="openpyxl") as writer:
    for index, row in rows.iterrows():
        prompt = get_prompt(zero_shot_template,row)
        for model, versions in ai_models.items():
            print(model,versions)
            for version in versions:   
                # print(start_row, index, model, version)
                match model:
                    case 'gpt':
                        response_text = call_gpt(version, prompt)
                    case 'claude':
                        response_text = call_claude(version, prompt)
                    # case 'gemini':
                        #TODO
                # write JSON result to excel file in SrpWN format  
                print(response_text)      
                clean_text = response_text.replace("```json", "").replace("```", "").strip()

                data = json.loads(clean_text)
                df = pd.DataFrame([data])
                df.to_excel(writer, sheet_name = version, startrow=index, index = False, header=False)
                
              
            
